## Description

This notebook defines a function `enrich_city_maker(df, city_col=...)` that takes in the Tarisio data and adds a small set of standardized geographic columns derived **GeoNames**.

The function `enrich_city_maker(...)' add the columns:
- `country_iso1`: 2-letter country code (e.g., `US`, `FR`, `GB`)
- `admin1_name`: first-order subdivision name (state/province/region)
- `admin2_name`: second-order subdivision name (county/district/province)
- `candidate_count`: number of GeoNames matches considered for that place
- `is_ambiguous`: `True` if `candidate_count > 1`

How matching is done:
- For each unique `city_maker` value, the code looks up matching entries in the offline GeoNames dump (`allCountries.txt`) using a normalized name key (case-insensitive, diacritics removed).
- US strings like `"City, ST"` force `country=US` and `admin1=ST`.
- Each GeoNames candidate gets a score based mainly on population:
  - `score = log(1 + population)`
  - plus a small bonus if the candidate is a populated place (`feature_class = 'P'`)
- If `print_ambiguous=True`, any names with multiple candidates are printed to the console for manual review.

In [4]:
import pandas as pd

df = pd.read_csv('cozio_sales_ALL.csv')
print(df.city_maker.value_counts())

city_maker
Paris                        8329
London                       6163
Mirecourt                    4337
Markneukirchen               2189
Cremona                      1478
Milan                        1109
Naples                       1007
Mittenwald                    741
Turin                         706
Mantua                        582
Florence                      509
Venice                        485
Rome                          364
Dresden                       354
Bologna                       331
Vienna                        318
Genoa                         313
Manchester                    310
Lyon                          290
Prague                        264
New York, NY                  252
Brussels                      219
Berlin                        213
Lille                         211
Modena                        148
Budapest                      144
The Hague                     143
Chicago, IL                   139
Ferrara                       132
Dub

In [ ]:
import math
import re
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import pandas as pd
import unicodedata

# Cleaning up names and parsing place strings

US_STATE_ABBR = set("""
AL AK AZ AR CA CO CT DE FL GA HI IA ID IL IN KS KY LA MA MD ME MI MN
MO MS MT NC ND NE NH NJ NM NV NY OH OK OR PA RI SC SD TN TX UT VA VT
WA WI WV WY DC
""".split())

US_STATE_NAME_TO_ABBR = {
    "alabama": "AL", "alaska": "AK", "arizona": "AZ", "arkansas": "AR",
    "california": "CA", "colorado": "CO", "connecticut": "CT", "delaware": "DE",
    "florida": "FL", "georgia": "GA", "hawaii": "HI", "idaho": "ID",
    "illinois": "IL", "indiana": "IN", "iowa": "IA", "kansas": "KS",
    "kentucky": "KY", "louisiana": "LA", "maine": "ME", "maryland": "MD",
    "massachusetts": "MA", "michigan": "MI", "minnesota": "MN", "mississippi": "MS",
    "missouri": "MO", "montana": "MT", "nebraska": "NE", "nevada": "NV",
    "new hampshire": "NH", "new jersey": "NJ", "new mexico": "NM", "new york": "NY",
    "north carolina": "NC", "north dakota": "ND", "ohio": "OH", "oklahoma": "OK",
    "oregon": "OR", "pennsylvania": "PA", "rhode island": "RI", "south carolina": "SC",
    "south dakota": "SD", "tennessee": "TN", "texas": "TX", "utah": "UT",
    "vermont": "VT", "virginia": "VA", "washington": "WA", "west virginia": "WV",
    "wisconsin": "WI", "wyoming": "WY", "district of columbia": "DC",
}

CHAR_MAP = str.maketrans({
    "ø": "o", "Ø": "O",
    "ł": "l", "Ł": "L",
    "đ": "d", "Đ": "D",
    "ð": "d", "Ð": "D",
    "þ": "th", "Þ": "Th",
    "æ": "ae", "Æ": "Ae",
    "œ": "oe", "Œ": "Oe",
})

def strip_accents(s: str) -> str:
    # Decompose + remove combining marks (accents/diacritics)
    s = unicodedata.normalize("NFKD", s)
    return "".join(ch for ch in s if not unicodedata.combining(ch))

def norm_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = s.translate(CHAR_MAP)
    s = strip_accents(s)
    s = s.casefold().strip()          # better than lower(); handles ß -> ss, etc.
    s = re.sub(r"[’`]", "'", s)       # normalize apostrophes
    s = re.sub(r"[\s\-]+", " ", s)    # collapse whitespace/hyphens
    s = re.sub(r"\s+", " ", s)
    return s

def parse_place(raw: str) -> Tuple[str, Optional[str], Optional[str], Optional[str]]:
    """
    Returns: (place_core, subregion_hint, state_hint, country_hint)
    - Handles:
        "City, ST" or "City,ST" -> state_hint=ST, country_hint='US'
        "City, StateName"       -> state_hint=abbr, country_hint='US'
        "Name (Qualifier)"      -> subregion_hint=Qualifier
        Otherwise:
        if comma present -> core before comma, subregion_hint after comma
    """
    if raw is None or (isinstance(raw, float) and math.isnan(raw)):
        return ("", None, None, None)

    s = str(raw).strip()
    if not s:
        return ("", None, None, None)

    # Parentheses qualifier: "Luxeuil (Haute-Saone)"
    m_paren = re.match(r"^(.*?)\s*\((.*?)\)\s*$", s)
    if m_paren:
        core = m_paren.group(1).strip()
        qual = m_paren.group(2).strip()
        return (core, qual or None, None, None)

    # Comma suffix: "New York, NY" or "Worthing, Sussex"
    if "," in s:
        left, right = s.rsplit(",", 1)
        left = left.strip()
        right_stripped = right.strip()

        # "City, ST"
        m_abbr = re.match(r"^([A-Za-z]{2})$", right_stripped)
        if m_abbr:
            st = m_abbr.group(1).upper()
            if st in US_STATE_ABBR:
                return (left, None, st, "US")

        # "City, StateName"
        right_norm = norm_text(right_stripped)
        if right_norm in US_STATE_NAME_TO_ABBR:
            st = US_STATE_NAME_TO_ABBR[right_norm]
            return (left, None, st, "US")

        # Otherwise treat comma part as a generic subregion hint
        return (left, right_stripped or None, None, None)

    return (s, None, None, None)


# GeoNames loading 
@dataclass(frozen=True)
class Candidate:
    geonameid: str
    name: str
    country: str
    admin1: str
    admin2: str
    feature_class: str
    feature_code: str
    population: int

def load_admin1(admin1_path: str) -> Dict[str, str]:
    # admin1CodesASCII.txt: "CC.ADM1\tName\t..."
    m = {}
    with open(admin1_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line.strip():
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 2:
                continue
            key = parts[0].strip()
            name = parts[1].strip()
            if key:
                m[key] = name
    return m

def load_admin2(admin2_path: str) -> Dict[str, str]:
    # admin2Codes.txt: "CC.ADM1.ADM2\tName\t..."
    m = {}
    with open(admin2_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line.strip():
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 2:
                continue
            key = parts[0].strip()
            name = parts[1].strip()
            if key:
                m[key] = name
    return m


def build_candidates_index(
    geonames_path: str,
    target_keys: set,
    include_admin_features: bool = True,
) -> Dict[str, List[Candidate]]:
    """
    Streams allCountries.txt once, collecting candidates only for keys in target_keys.
    Pass 1: match on (normalized) asciiname and name.
    Pass 2: (optional) match on alternatenames only for keys still unmatched.

    Keeps:
      - feature_class 'P' (populated places)
      - plus (optional) feature_class 'A' with admin codes (ADM1/ADM2/...)
    """
    allowed_A_codes = {"ADM1", "ADM2", "ADM3", "ADM4", "ADMD", "ADM5"}

    cand_by_key: Dict[str, List[Candidate]] = {k: [] for k in target_keys}

    def maybe_add(key: str, parts: List[str]) -> None:
        # columns per GeoNames readme:
        # 0 geonameid, 1 name, 2 asciiname, 3 alternatenames, 6 fclass, 7 fcode,
        # 8 country, 10 admin1, 11 admin2, 14 population
        fclass = parts[6]
        fcode = parts[7]
        if fclass == "P":
            pass
        elif include_admin_features and fclass == "A" and fcode in allowed_A_codes:
            pass
        else:
            return

        try:
            pop = int(parts[14]) if parts[14] else 0
        except ValueError:
            pop = 0

        cand_by_key[key].append(
            Candidate(
                geonameid=parts[0],
                name=parts[1],
                country=parts[8],
                admin1=parts[10],
                admin2=parts[11],
                feature_class=fclass,
                feature_code=fcode,
                population=pop,
            )
        )

    # pass 1: asciiname/name
    with open(geonames_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line.strip():
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 15:
                continue

            key_ascii = norm_text(parts[2])
            if key_ascii in target_keys:
                maybe_add(key_ascii, parts)

            key_name = norm_text(parts[1])
            if key_name in target_keys and key_name != key_ascii:
                maybe_add(key_name, parts)

    # find unmatched keys
    unmatched = {k for k, v in cand_by_key.items() if len(v) == 0}
    if not unmatched:
        return cand_by_key

    #  pass 2: alternatenames for unmatched only
    with open(geonames_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line.strip():
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 15:
                continue

            alts = parts[3]
            if not alts:
                continue

            # quick skip: if we already matched all, stop
            if not unmatched:
                break

            # Normalize alternatenames and test membership in unmatched.
            # This is the expensive part; we only do it for the remaining unmatched keys.
            for alt in alts.split(","):
                k = norm_text(alt)
                if k in unmatched:
                    maybe_add(k, parts)

    return cand_by_key


# Main function to enrich city_maker with GeoNames data

def enrich_city_maker(
    df: pd.DataFrame,
    city_col: str = "city_maker",
    geonames_path: str = "allCountries.txt",
    admin1_path: str = "admin1CodesASCII.txt",
    admin2_path: str = "admin2Codes.txt",
    print_ambiguous: bool = True,
    top_k_print: int = 5,
) -> pd.DataFrame:
    """
    Adds columns:
      - country_iso1 (GeoNames country_code, e.g., 'US', 'FR', 'GB')
      - admin1_name
      - admin2_name
      - candidate_count
      - is_ambiguous

    Prints ambiguous place strings to console with top candidate suggestions.
    Handles missing/blank city_maker gracefully.
    """
    out = df.copy()

    if city_col not in out.columns:
        raise ValueError(f"Column '{city_col}' not found in dataframe.")

    # Parse unique places (skip NaN/empty)
    uniques = pd.Series(out[city_col].dropna().unique())
    parsed = []
    for raw in uniques.tolist():
        core, sub_hint, st_hint, c_hint = parse_place(raw)
        key = norm_text(core)
        if key:
            parsed.append((raw, core, sub_hint, st_hint, c_hint, key))

    if not parsed:
        # Add empty columns and return
        out["country_iso1"] = pd.NA
        out["admin1_name"] = pd.NA
        out["admin2_name"] = pd.NA
        out["candidate_count"] = pd.NA
        out["is_ambiguous"] = False
        return out

    places = pd.DataFrame(parsed, columns=["place_raw", "place_core", "subregion_hint", "state_hint", "country_hint", "place_key"])

    # Load decoding maps
    admin1_map = load_admin1(admin1_path)
    admin2_map = load_admin2(admin2_path)

    # Build candidate index for all needed keys
    target_keys = set(places["place_key"].unique())
    cand_by_key = build_candidates_index(geonames_path, target_keys, include_admin_features=True)

    # For each unique raw place, pick best candidate using hints 
    resolved_rows = []
    ambiguous_reports = []

    def score_candidate(c: Candidate) -> float:
        s = math.log1p(max(c.population, 0))
        if c.feature_class == "P":
            s += 2.0
        return s

    for _, r in places.iterrows():
        raw = r["place_raw"]
        key = r["place_key"]
        st_hint = r["state_hint"]
        c_hint = r["country_hint"]

        candidates = cand_by_key.get(key, [])
        # Apply country/state hints when present
        filtered = candidates

        if c_hint:
            filtered = [c for c in filtered if c.country == c_hint]

        if st_hint:
            # GeoNames admin1_code for US states aligns with keys in admin1CodesASCII like "US.NY"
            filtered2 = [c for c in filtered if c.country == "US" and c.admin1 == st_hint]
            if filtered2:
                filtered = filtered2

        # If filtering removed everything, fall back to unfiltered 
        if not filtered:
            filtered = candidates

        scored = [(score_candidate(c), c) for c in filtered]
        scored.sort(key=lambda t: t[0], reverse=True)

        cand_count = len(filtered)
        is_amb = cand_count > 1

        if scored:
            best_c = scored[0][1]
            country_iso1 = best_c.country or pd.NA

            a1_key = f"{best_c.country}.{best_c.admin1}" if best_c.country and best_c.admin1 else None
            admin1_name = admin1_map.get(a1_key, pd.NA) if a1_key else pd.NA

            a2_key = f"{best_c.country}.{best_c.admin1}.{best_c.admin2}" if best_c.country and best_c.admin1 and best_c.admin2 else None
            admin2_name = admin2_map.get(a2_key, pd.NA) if a2_key else pd.NA
        else:
            # no match at all
            country_iso1 = pd.NA
            admin1_name = pd.NA
            admin2_name = pd.NA
            cand_count = 0
            is_amb = False

        resolved_rows.append({
            "place_raw": raw,
            "country_iso1": country_iso1,
            "admin1_name": admin1_name,
            "admin2_name": admin2_name,
            "candidate_count": cand_count,
            "is_ambiguous": bool(is_amb),
        })

        if print_ambiguous and is_amb and scored:
            # Print a compact report with top options
            top = scored[:top_k_print]
            opts = []
            for s_val, c in top:
                a1 = admin1_map.get(f"{c.country}.{c.admin1}", "")
                a2 = admin2_map.get(f"{c.country}.{c.admin1}.{c.admin2}", "")
                opts.append((s_val, c.name, c.country, a1, a2, c.population))
            ambiguous_reports.append((raw, cand_count, opts))

    mapping = pd.DataFrame(resolved_rows).drop_duplicates("place_raw")
    out = out.merge(mapping, left_on=city_col, right_on="place_raw", how="left").drop(columns=["place_raw"])

    # Ensure rows with missing city_maker remain clean (merge will keep NaNs)
    out["is_ambiguous"] = out["is_ambiguous"].fillna(False).astype(bool)

    # Console printing
    if print_ambiguous and ambiguous_reports:
        print("\n--- Ambiguous place names (review recommended) ---")
        for raw, n, opts in ambiguous_reports:
            print(f"\nAMBIGUOUS: '{raw}' (candidates={n})")
            for s_val, name, cc, a1, a2, pop in opts:
                a1_txt = f" | admin1={a1}" if a1 else ""
                a2_txt = f" | admin2={a2}" if a2 else ""
                print(f"  score={s_val:0.3f} | {name} | {cc}{a1_txt}{a2_txt} | pop={pop}")

    return out

In [8]:

df2 = enrich_city_maker(
    df,
    city_col="city_maker",
    geonames_path="Data/Geo_Data/allCountries.txt",
    admin1_path="Data/Geo_Data/admin1CodesASCII.txt",
    admin2_path="Data/Geo_Data/admin2Codes.txt",
    print_ambiguous=True,
)
# df2.to_csv("cozio_sales_grouped_by_region.csv", index=False)


--- Ambiguous place names (review recommended) ---

AMBIGUOUS: 'Wallgau' (candidates=2)
  score=9.277 | Wallgau | DE | admin1=Bavaria | admin2=Upper Bavaria | pop=1446
  score=7.324 | Wallgau | DE | admin1=Bavaria | admin2=Upper Bavaria | pop=1516

AMBIGUOUS: 'Paris' (candidates=59)
  score=16.576 | Paris | FR | admin1=Île-de-France | admin2=Paris | pop=2138551
  score=15.293 | Paris | FR | admin1=Île-de-France | admin2=Paris | pop=4380654
  score=14.600 | Paris | FR | admin1=Île-de-France | admin2=Paris | pop=2190327
  score=14.533 | Paris | FR | admin1=Île-de-France | admin2=Paris | pop=2048472
  score=12.118 | Paris | US | admin1=Texas | admin2=Lamar County | pop=24782

AMBIGUOUS: 'Mirecourt' (candidates=2)
  score=10.854 | Mirecourt | FR | admin1=Grand Est | admin2=Vosges | pop=6998
  score=8.573 | Mirecourt | FR | admin1=Grand Est | admin2=Vosges | pop=5285

AMBIGUOUS: 'London' (candidates=34)
  score=18.009 | London | GB | admin1=England | admin2=Greater London | pop=8961989
  s

# Manually Checking Candidates #
Most of the town names have multiple matches. Most of them are safe to assume they are the top scored candidate (e.g. when the other candidates all have population 0 or when some towns are listed twice in the geonames data). The more ambiguous ones I have to crosscheck with the maker data manually to see what country they were based in. So I rewrite the function so that:
- The candidate with the highest score is selected.
- For some names, hard-coded overrides force the intended country and/or lookup name; if no match exists under the forced country, the result is left blank.

In [ ]:
import math
import re
import unicodedata
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple, Union

import pandas as pd

CHAR_MAP = str.maketrans({
    "ø": "o", "Ø": "O",
    "ł": "l", "Ł": "L",
    "đ": "d", "Đ": "D",
    "ð": "d", "Ð": "D",
    "þ": "th", "Þ": "Th",
    "æ": "ae", "Æ": "Ae",
    "œ": "oe", "Œ": "Oe",
})

def strip_accents(s: str) -> str:
    s = unicodedata.normalize("NFKD", s)
    return "".join(ch for ch in s if not unicodedata.combining(ch))

def norm_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = s.translate(CHAR_MAP)
    s = strip_accents(s)
    s = s.casefold().strip()
    s = re.sub(r"[’`]", "'", s)        # normalize apostrophes
    s = re.sub(r"[\s\-]+", " ", s)     # collapse whitespace/hyphens
    s = re.sub(r"\s+", " ", s)
    return s


US_STATE_ABBR = set("""
AL AK AZ AR CA CO CT DE FL GA HI IA ID IL IN KS KY LA MA MD ME MI MN
MO MS MT NC ND NE NH NJ NM NV NY OH OK OR PA RI SC SD TN TX UT VA VT
WA WI WV WY DC
""".split())

US_STATE_NAME_TO_ABBR = {
    "alabama": "AL", "alaska": "AK", "arizona": "AZ", "arkansas": "AR",
    "california": "CA", "colorado": "CO", "connecticut": "CT", "delaware": "DE",
    "florida": "FL", "georgia": "GA", "hawaii": "HI", "idaho": "ID",
    "illinois": "IL", "indiana": "IN", "iowa": "IA", "kansas": "KS",
    "kentucky": "KY", "louisiana": "LA", "maine": "ME", "maryland": "MD",
    "massachusetts": "MA", "michigan": "MI", "minnesota": "MN", "mississippi": "MS",
    "missouri": "MO", "montana": "MT", "nebraska": "NE", "nevada": "NV",
    "new hampshire": "NH", "new jersey": "NJ", "new mexico": "NM", "new york": "NY",
    "north carolina": "NC", "north dakota": "ND", "ohio": "OH", "oklahoma": "OK",
    "oregon": "OR", "pennsylvania": "PA", "rhode island": "RI", "south carolina": "SC",
    "south dakota": "SD", "tennessee": "TN", "texas": "TX", "utah": "UT",
    "vermont": "VT", "virginia": "VA", "washington": "WA", "west virginia": "WV",
    "wisconsin": "WI", "wyoming": "WY", "district of columbia": "DC",
}

def parse_place(raw: str) -> Tuple[str, Optional[str], Optional[str], Optional[str]]:
    """
    Returns: (place_core, subregion_hint, state_hint, country_hint)
    - "City, ST" -> state_hint=ST, country_hint='US'
    - "City, StateName" -> state_hint=abbr, country_hint='US'
    - "Name (Qualifier)" -> place_core=Name, subregion_hint=Qualifier
    - Otherwise: place_core=raw
    """
    if raw is None or (isinstance(raw, float) and math.isnan(raw)):
        return ("", None, None, None)

    s = str(raw).strip()
    if not s:
        return ("", None, None, None)

    m_paren = re.match(r"^(.*?)\s*\((.*?)\)\s*$", s)
    if m_paren:
        core = m_paren.group(1).strip()
        qual = m_paren.group(2).strip()
        return (core, qual or None, None, None)

    if "," in s:
        left, right = s.rsplit(",", 1)
        left = left.strip()
        right_stripped = right.strip()

        m_abbr = re.match(r"^([A-Za-z]{2})$", right_stripped)
        if m_abbr:
            st = m_abbr.group(1).upper()
            if st in US_STATE_ABBR:
                return (left, None, st, "US")

        right_norm = norm_text(right_stripped)
        if right_norm in US_STATE_NAME_TO_ABBR:
            st = US_STATE_NAME_TO_ABBR[right_norm]
            return (left, None, st, "US")

        return (left, right_stripped or None, None, None)

    return (s, None, None, None)


@dataclass(frozen=True)
class Candidate:
    geonameid: str
    name: str
    asciiname: str
    country: str
    admin1: str
    admin2: str
    feature_class: str
    feature_code: str
    population: int

def load_admin1(admin1_path: str) -> Dict[str, str]:
    # admin1CodesASCII.txt: "CC.ADM1\tName\t..."
    m = {}
    with open(admin1_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line.strip():
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 2:
                continue
            key = parts[0].strip()
            name = parts[1].strip()
            if key:
                m[key] = name
    return m

def load_admin2(admin2_path: str) -> Dict[str, str]:
    # admin2Codes.txt: "CC.ADM1.ADM2\tName\t..."
    m = {}
    with open(admin2_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line.strip():
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 2:
                continue
            key = parts[0].strip()
            name = parts[1].strip()
            if key:
                m[key] = name
    return m

def build_candidates_index(
    geonames_path: str,
    target_keys: set,
    include_admin_features: bool = True,
) -> Dict[str, List[Candidate]]:
    """
    Streams allCountries.txt and collects candidates only for keys in target_keys.
    Pass 1 matches on normalized 'name' and 'asciiname'.
    Pass 2 (optional) matches on alternatenames only for keys still unmatched.
    """
    allowed_A_codes = {"ADM1", "ADM2", "ADM3", "ADM4", "ADMD", "ADM5"}

    cand_by_key: Dict[str, List[Candidate]] = {k: [] for k in target_keys}

    def maybe_add(key: str, parts: List[str]) -> None:
        # GeoNames columns (0-based):
        # 0 geonameid, 1 name, 2 asciiname, 3 alternatenames, 6 fclass, 7 fcode,
        # 8 country, 10 admin1, 11 admin2, 14 population
        fclass = parts[6]
        fcode = parts[7]
        if fclass == "P":
            pass
        elif include_admin_features and fclass == "A" and fcode in allowed_A_codes:
            pass
        else:
            return

        try:
            pop = int(parts[14]) if parts[14] else 0
        except ValueError:
            pop = 0

        cand_by_key[key].append(
            Candidate(
                geonameid=parts[0],
                name=parts[1],
                asciiname=parts[2],
                country=parts[8],
                admin1=parts[10],
                admin2=parts[11],
                feature_class=fclass,
                feature_code=fcode,
                population=pop,
            )
        )

    # pass 1 
    with open(geonames_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line.strip():
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 15:
                continue

            k_ascii = norm_text(parts[2])
            if k_ascii in target_keys:
                maybe_add(k_ascii, parts)

            k_name = norm_text(parts[1])
            if k_name in target_keys and k_name != k_ascii:
                maybe_add(k_name, parts)

    unmatched = {k for k, v in cand_by_key.items() if len(v) == 0}
    if not unmatched:
        return cand_by_key

    # pass 2 (alternatenames for remaining unmatched only)
    with open(geonames_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line.strip():
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 15:
                continue
            alts = parts[3]
            if not alts:
                continue
            if not unmatched:
                break

            for alt in alts.split(","):
                k = norm_text(alt)
                if k in unmatched:
                    maybe_add(k, parts)

    return cand_by_key


OVERRIDES: Dict[str, Dict[str, Union[str, List[str], bool, List[str]]]] = {
    # special remaps / forced countries
    "gerona":      {"country": "IT", "lookup": ["fosso gerona"]},
    "mantua":      {"country": "IT", "lookup": ["mantova", "mantua"], "prefer_class": "A", "prefer_codes": ["ADM2", "ADM1"]},
    "cheshire":    {"country": "GB", "prefer_class": "A"},
    "padua":       {"country": "IT", "lookup": ["padova", "padua"]},
    "beaconsfield":{"country": "GB"},
    "lugo":        {"country": "IT"},
    "san marino":  {"country": "SM"},
    "devonport":   {"country": "GB"},
    "tournay":     {"country": "BE", "lookup": ["tournai", "tournay"]},
    "berne":       {"country": "CH", "lookup": ["bern", "berne"]},
    "antwerp":     {"country": "BE"},
    "valencia":    {"country": "ES"},
    "kew":         {"country": "GB"},
    "gand":        {"country": "BE", "lookup": ["gent", "ghent", "gand"]},
    "salo":        {"country": "IT"},
    "surrey":      {"country": "GB", "prefer_class": "A"},
    "hanwell":     {"country": "GB"},
    "kensington":  {"country": "GB"},
    "hull":        {"country": "GB"},
    "bernal":      {"country": "AR"},
    "moravia":     {"country": "CZ", "prefer_class": "A"},
    "orleans":     {"country": "FR"},
    "aquila":      {"country": "IT", "lookup": ["l'aquila", "laquila", "l aquila", "aquila"]},
    "leith":       {"country": "GB"},
    "pesth":       {"country": "HU", "lookup": ["pest", "pesth"]},
    "leonardo":    {"country": "IT"},
    "geneva":      {"country": "CH"},
    "san remo":    {"country": "IT", "lookup": ["sanremo", "san remo"]},
    "cumberland":  {"country": "GB", "prefer_class": "A"},
    "lancaster":   {"country": "US", "state_hint": "PA"},  # Lancaster, Pennsylvania
    "alva":        {"country": "GB"},
    "hanley":      {"country": "GB"},

    # force blank
    "franklin":    {"blank": True},
}


def enrich_city_maker(
    df: pd.DataFrame,
    city_col: str = "city_maker",
    geonames_path: str = "allCountries.txt",
    admin1_path: str = "admin1CodesASCII.txt",
    admin2_path: str = "admin2Codes.txt",
    print_ambiguous: bool = True,
    top_k_print: int = 5,
) -> pd.DataFrame:
    """
    Adds:
      - country_iso1
      - admin1_name
      - admin2_name
      - candidate_count
      - is_ambiguous

    Default selection = highest score (population-driven).
    Overrides applied for specified place_core keys in OVERRIDES.
    If an override forces a country and no candidate exists in that country, outputs blanks.
    """
    out = df.copy()
    if city_col not in out.columns:
        raise ValueError(f"Column '{city_col}' not found.")

    # Prepare unique places
    uniques = pd.Series(out[city_col].dropna().unique())
    parsed = []
    for raw in uniques.tolist():
        core, sub_hint, st_hint, c_hint = parse_place(raw)
        core_key = norm_text(core)
        if core_key:
            parsed.append((raw, core, core_key, st_hint, c_hint))

    # If no valid places, just add empty cols
    if not parsed:
        out["country_iso1"] = pd.NA
        out["admin1_name"] = pd.NA
        out["admin2_name"] = pd.NA
        out["candidate_count"] = pd.NA
        out["is_ambiguous"] = False
        return out

    places = pd.DataFrame(parsed, columns=["place_raw", "place_core", "place_key", "state_hint", "country_hint"])

    # Build target keys: include override lookup keys too
    target_keys = set(places["place_key"].unique())
    for k, spec in OVERRIDES.items():
        lookup = spec.get("lookup", None)
        if isinstance(lookup, str):
            target_keys.add(norm_text(lookup))
        elif isinstance(lookup, list):
            for name in lookup:
                target_keys.add(norm_text(name))

    # Load decoding maps
    admin1_map = load_admin1(admin1_path)
    admin2_map = load_admin2(admin2_path)

    # Build candidate index for all needed keys
    cand_by_key = build_candidates_index(geonames_path, target_keys, include_admin_features=True)

    def score_candidate(c: Candidate, prefer_class: Optional[str], prefer_codes: Optional[List[str]]) -> float:
        s = math.log1p(max(c.population, 0))
        # mild preference toward populated places by default
        if c.feature_class == "P":
            s += 2.0
        # optional preference requested by override (e.g., choose ADM2 over city)
        if prefer_class and c.feature_class == prefer_class:
            s += 3.0
        if prefer_codes and c.feature_code in set(prefer_codes):
            s += 2.0
        return s

    resolved_rows = []
    ambiguous_reports = []

    for _, r in places.iterrows():
        raw = r["place_raw"]
        core_key = r["place_key"]

        # base hints from parsing
        st_hint = r["state_hint"]
        c_hint = r["country_hint"]

        # override spec (by normalized core)
        spec = OVERRIDES.get(core_key, {})
        if spec.get("blank", False):
            resolved_rows.append({
                "place_raw": raw,
                "country_iso1": pd.NA,
                "admin1_name": pd.NA,
                "admin2_name": pd.NA,
                "candidate_count": 0,
                "is_ambiguous": False,
            })
            continue

        # apply override hints
        forced_country = spec.get("country", None) or c_hint
        forced_state   = spec.get("state_hint", None) or st_hint
        prefer_class   = spec.get("prefer_class", None)
        prefer_codes   = spec.get("prefer_codes", None)

        # determine which lookup keys to try (override lookup list, else the core key)
        lookup_names = spec.get("lookup", None)
        if isinstance(lookup_names, str):
            lookup_keys = [norm_text(lookup_names)]
        elif isinstance(lookup_names, list):
            lookup_keys = [norm_text(x) for x in lookup_names]
        else:
            lookup_keys = [core_key]

        # gather candidates across lookup keys (dedupe by geonameid)
        candidates = []
        seen = set()
        for lk in lookup_keys:
            for c in cand_by_key.get(lk, []):
                if c.geonameid not in seen:
                    seen.add(c.geonameid)
                    candidates.append(c)

        # if override forces a country, we must not fall back outside it
        if forced_country:
            candidates_in_country = [c for c in candidates if c.country == forced_country]
        else:
            candidates_in_country = candidates

        # if forced_state is present (only reliable for US), filter further if possible
        filtered = candidates_in_country
        if forced_state:
            filtered2 = [c for c in filtered if c.country == "US" and c.admin1 == forced_state]
            if filtered2:
                filtered = filtered2

        # If forced country was specified and nothing matches, return blank 
        if forced_country and not filtered:
            resolved_rows.append({
                "place_raw": raw,
                "country_iso1": pd.NA,
                "admin1_name": pd.NA,
                "admin2_name": pd.NA,
                "candidate_count": 0,
                "is_ambiguous": False,
            })
            continue

        # If not forced country and filtering yields nothing, fall back to all candidates
        if not forced_country and not filtered:
            filtered = candidates

        cand_count = len(filtered)
        is_amb = cand_count > 1

        if filtered:
            scored = [(score_candidate(c, prefer_class, prefer_codes), c) for c in filtered]
            scored.sort(key=lambda t: t[0], reverse=True)
            best = scored[0][1]

            country_iso1 = best.country or pd.NA
            a1_key = f"{best.country}.{best.admin1}" if best.country and best.admin1 else None
            admin1_name = admin1_map.get(a1_key, pd.NA) if a1_key else pd.NA

            a2_key = f"{best.country}.{best.admin1}.{best.admin2}" if best.country and best.admin1 and best.admin2 else None
            admin2_name = admin2_map.get(a2_key, pd.NA) if a2_key else pd.NA

            if print_ambiguous and is_amb and core_key not in OVERRIDES:
                top = scored[:top_k_print]
                opts = []
                for s_val, c in top:
                    a1 = admin1_map.get(f"{c.country}.{c.admin1}", "")
                    a2 = admin2_map.get(f"{c.country}.{c.admin1}.{c.admin2}", "")
                    opts.append((s_val, c.name, c.asciiname, c.country, a1, a2, c.feature_class, c.feature_code, c.population))
                ambiguous_reports.append((raw, cand_count, opts))
        else:
            country_iso1 = pd.NA
            admin1_name = pd.NA
            admin2_name = pd.NA

        resolved_rows.append({
            "place_raw": raw,
            "country_iso1": country_iso1,
            "admin1_name": admin1_name,
            "admin2_name": admin2_name,
            "candidate_count": cand_count,
            "is_ambiguous": bool(is_amb),
        })

    mapping = pd.DataFrame(resolved_rows).drop_duplicates("place_raw")
    out = out.merge(mapping, left_on=city_col, right_on="place_raw", how="left").drop(columns=["place_raw"])
    out["is_ambiguous"] = out["is_ambiguous"].fillna(False).astype(bool)

    if print_ambiguous and ambiguous_reports:
        print("\n--- Ambiguous place names (not covered by overrides) ---")
        for raw, n, opts in ambiguous_reports:
            print(f"\nAMBIGUOUS: '{raw}' (candidates={n})")
            for s_val, name, asciiname, cc, a1, a2, fcl, fcd, pop in opts:
                a1_txt = f" | admin1={a1}" if a1 else ""
                a2_txt = f" | admin2={a2}" if a2 else ""
                print(f"  score={s_val:0.3f} | {name} ({asciiname}) | {cc}{a1_txt}{a2_txt} | {fcl}/{fcd} | pop={pop}")

    return out


In [11]:

df2 = enrich_city_maker(df, city_col="city_maker",
                        geonames_path="Data/Geo_Data/allCountries.txt",
                        admin1_path="Data/Geo_Data/admin1CodesASCII.txt",
                        admin2_path="Data/Geo_Data/admin2Codes.txt",
                        print_ambiguous=True)


--- Ambiguous place names (not covered by overrides) ---

AMBIGUOUS: 'Grand Rapids, MI' (candidates=2)
  score=14.181 | Grand Rapids (Grand Rapids) | US | admin1=Michigan | admin2=Kent County | P/PPLA2 | pop=195097
  score=2.000 | Grand Rapids (Grand Rapids) | US | admin1=Michigan | admin2=Ontonagon County | P/PPL | pop=0

AMBIGUOUS: 'Santa Rosa, CA' (candidates=4)
  score=14.090 | Santa Rosa (Santa Rosa) | US | admin1=California | admin2=Sonoma County | P/PPLA2 | pop=178127
  score=0.000 | Santa Rosa (Santa Rosa) | US | admin1=California | admin2=San Luis Obispo County | A/ADMD | pop=0
  score=0.000 | Santa Rosa (Santa Rosa) | US | admin1=California | admin2=Santa Barbara County | A/ADMD | pop=0
  score=0.000 | Santa Rosa (Santa Rosa) | US | admin1=California | admin2=Riverside County | A/ADMD | pop=0

AMBIGUOUS: 'Brunswick, IN' (candidates=2)
  score=2.000 | Brunswick (Brunswick) | US | admin1=Indiana | admin2=Clay County | P/PPL | pop=0
  score=2.000 | Brunswick (Brunswick) | US | 

In [ ]:
df2.to_csv("cozio_sales_ALL_grouped_by_regions.csv", index=False)

In [1]:
df2.columns()

NameError: name 'df2' is not defined